# Hypothesis - Test Energy and Popularity

Test whether there is a statistically significant relationship between energy and popularity.

- Check the relevant data distributions
- Choose an appropriate statistical test
- Run the statistical test
- Record the result and p-value
- Interpret the result
- State the conclusion for the null hypothesis



## Inputs

CSV file used:

spotifydataset_Visualisation.csv


## Outputs

Plots for testing validity of each hypothesis before considering visualisation


## Additional Comments

Developed an experimental ETL library which is in this project (modETL_library.py) 
Has lots of cool features so will be interesting to see how it works "in the field"

In [1]:
# import libraries
import os
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import kruskal

# added by me for visualisation fine tuning
from matplotlib.ticker import MultipleLocator
from scipy.stats import f_oneway
from scipy.stats import pearsonr
from scipy.stats import spearmanr
from scipy.stats import ttest_ind

# added by me for plotly.express visualisation issue
import nbformat

# below solution provided by chatGPT
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root / "assets" / "python_files") not in sys.path:
    sys.path.insert(0, str(project_root / "assets" / "python_files"))

import modGlobal
import modETL_Library as modETL
# end solution provided by chatGPT

# Section 1 - Initalisation

## Intitialise All Variables To Be Used In Global Stack 

In [2]:
# DataFrame variables for EDA
dfSpotify_DataSet_Work = None
dfSpotify_DataSet_Temp = None
dfSpotify_DataSet_Temp1 = None
dfSpotify_DataSet_Temp2 = None
dfSpotify_DataSet_Merged = None

# stores current directory
strCurrentDir = ""

# other vars
dictDataFrames = dict()
fig = None
axis = None
fltShapiroStat = 0.0
fltShapiroP = 0.0
fltMannWhitneyU = 0.0
fltMannWhitneyUStat = 0.0
fltMannWhitneyUp = 0.0
fltTStatistic = 0.0
fltPValue = 0.0


## Set Current Directory To Base Project Directory

In [3]:
# get project directory - default is jupyter notebook sub folder as that is where this file is located!
# so move back one to the project root path
# Source - https://stackoverflow.com/a/17726833
# Posted by chimpsarehungry
# Retrieved 2026-07-05, License - CC BY-SA 3.0

# get current folder
strCurrentDir = os.getcwd()

# is the last part of the path the project directory?
if not strCurrentDir.endswith(modGlobal.CNST_STR_PROJECT_DIR):
    # get current working directory and move back one to the project root path
    strCurrentDir = os.path.normpath(os.getcwd() + os.sep + os.pardir)
    os.chdir(os.path.dirname(strCurrentDir))
    # change directory
    os.chdir(strCurrentDir)

# confirm current directory is project directory
print(f"Current Directory: \n {os.getcwd()}")

Current Directory: 
 /Users/sahraosman/Documents/vscode-projects/Spotify Music Trend Analysis


# Section 2

- Read csv file
- Look at hypothesis validity
- Use plot(s) to validate findings

## Read csv File Into Variable For Processing

In [4]:
# read csv file into DataFrame
dictDataFrames = modETL.funcReadVisualisationFilesReturnDictionary()
dfSpotify_DataSet = dictDataFrames["spotifydataset_Visualisation.csv"]

# create copy of the original DataFrame to work with
dfSpotify_DataSet_Work = dfSpotify_DataSet.copy()

2 csv Files Read Into DataFrames

DataFrames Created:
spotifydataset_Visualisation.csv
spotify_dashboard_data.csv




# Check Relevant Data Distributions Between Energy And Popularity

Look at:

- Skew
- Kurtosis
- Q-Q plot
- Correlations

In [5]:
# get skew and kurtosis and Q-Q plots for energy and popularity columns

# get data minus duplicates
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(
    by="popularity", ascending=False
).drop_duplicates(subset=["artists", "album_name", "track_name"], keep="first")

# only need energy and popularity columns for skew and kurtosis
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp[["energy", "popularity"]]

modETL.funcGetColumnSkew(dfSpotify_DataSet_Temp)

DataFrame Skew Values Per Column:
energy               Skew: -0.56 - Highly Negatively Skewed Kurtosis: -0.61
popularity           Skew: 0.07 - Approximately Symmetrical Kurtosis: -0.77

Q-Q Plot For Column: energy


<Figure size 1200x600 with 1 Axes>



Q-Q Plot For Column: popularity


<Figure size 1200x600 with 1 Axes>

# Observations

Energy is highly skewed and popularity is around central tendancy, which technically implies no correlation 

# Check For Correlation

In [6]:
# see correlation between energy and popularity

print(dfSpotify_DataSet_Work[["energy", "popularity"]].corr())

              energy  popularity
energy      1.000000    0.001053
popularity  0.001053    1.000000


# Observations

Energy is 0.001053 which is technically zero which suggests there is no correlation between energy and popularity
Energy and popularity vs themselves is always 1 so we can ignore that!

# Statistical Tests

Basically rather than select one I think *might* be suitable instead I shall throw the proverbial kitchen sink at it!

# Shapiro-Wilk Test

In [7]:
# Shapiro-Wilk Test for energy column

# get records without duplicates
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(
    by="popularity", ascending=False
).drop_duplicates(subset=["artists", "album_name", "track_name"], keep="first")

# Shapiro-Wilk Test for energy column using 3000 sample size due to data size
dfSpotify_DataSet_Temp1 = dfSpotify_DataSet_Temp["energy"].sample(3000, random_state=42)
fltShapiroStat, fltShapiroP = stats.shapiro(dfSpotify_DataSet_Temp1)
print(
    f"Shapiro-Wilk p-value (Probabilty) Based On Random Sample Of 3000 Records: {fltShapiroP:.4f}"
)

# Shapiro-Wilk Test for popularity column
dfSpotify_DataSet_Temp2 = dfSpotify_DataSet_Temp["popularity"].sample(
    3000, random_state=42
)
fltShapiroStat, fltShapiroP = stats.shapiro(dfSpotify_DataSet_Temp2)
print(
    f"Shapiro-Wilk p-value (Probabilty) Based On Random Sample Of 3000 Records: {fltShapiroP:.4f}"
)

Shapiro-Wilk p-value (Probabilty) Based On Random Sample Of 3000 Records: 0.0000
Shapiro-Wilk p-value (Probabilty) Based On Random Sample Of 3000 Records: 0.0000


# Observations

As expected results are zero, however still a test worth doing

# Non Parametric Tests

In [8]:
# get records without duplicates

# do non parametric test for energy column
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(
    by="popularity", ascending=False
).drop_duplicates(subset=["artists", "album_name", "track_name"], keep="first")[
    "energy"
]

dfSpotify_DataSet_Temp1 = dfSpotify_DataSet_Temp.sample(1000, random_state=42)
dfSpotify_DataSet_Temp2 = dfSpotify_DataSet_Temp.sample(1000, random_state=42)

fltMannWhitneyUStat, fltMannWhitneyUp = stats.mannwhitneyu(
    dfSpotify_DataSet_Temp1, dfSpotify_DataSet_Temp2
)
print("Non Parametric Test Results For Energy Column")
print("*" * len("Non Parametric Test Results For Energy Column"))
print(
    f"Mann-Whitney U Test Results: Stat={fltMannWhitneyUStat}, P={fltMannWhitneyUp:.4f}"
)
print()

# do non parametric test for popularity column
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(
    by="popularity", ascending=False
).drop_duplicates(subset=["artists", "album_name", "track_name"], keep="first")[
    "popularity"
]

dfSpotify_DataSet_Temp1 = dfSpotify_DataSet_Temp.sample(1000, random_state=42)
dfSpotify_DataSet_Temp2 = dfSpotify_DataSet_Temp.sample(1000, random_state=42)

fltMannWhitneyUStat, fltMannWhitneyUp = stats.mannwhitneyu(
    dfSpotify_DataSet_Temp1, dfSpotify_DataSet_Temp2
)
print("Non Parametric Test Results For Popularity Column")
print("*" * len("Non Parametric Test Results For Popularity Column"))
print(
    f"Mann-Whitney U Test Results: Stat={fltMannWhitneyUStat}, P={fltMannWhitneyUp:.4f}"
)
print()


# do non parametric test for energy column
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(
    by="popularity", ascending=False
).drop_duplicates(subset=["artists", "album_name", "track_name"], keep="first")

dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp[["popularity", "energy"]]

# run Spearman correlation test for energy and popularity columns
fltSpearmanCorr, fltSpearmanP = spearmanr(
    dfSpotify_DataSet_Temp["energy"], dfSpotify_DataSet_Temp["popularity"]
)
print("Spearman Correlation Test Results For Energy Column Against Popularity")
print(
    "*" * len("Spearman Correlation Test Results For Energy Column Against Popularity")
)
print(
    f"Spearman Correlation Coefficient: {fltSpearmanCorr:.4f}, P-value: {fltSpearmanP:.4f}"
)
print()


Non Parametric Test Results For Energy Column
*********************************************
Mann-Whitney U Test Results: Stat=500000.0, P=1.0000

Non Parametric Test Results For Popularity Column
*************************************************
Mann-Whitney U Test Results: Stat=500000.0, P=1.0000

Spearman Correlation Test Results For Energy Column Against Popularity
**********************************************************************
Spearman Correlation Coefficient: -0.0166, P-value: 0.0000



# Observations

P-value of 1 for both single column non parametric tests means no rejection of hypothesis zero
P-value of 0 for the combined non parametric test means no relationship

# Parametric Tests

In [9]:
# Parametric Individual T-Test for energy column set equal_var to false as not assuming population variances are equal

# run T-Test for energy column
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(
    by="popularity", ascending=False
).drop_duplicates(subset=["artists", "album_name", "track_name"], keep="first")[
    "energy"
]

dfSpotify_DataSet_Temp1 = dfSpotify_DataSet_Temp.sample(1000, random_state=42)
dfSpotify_DataSet_Temp2 = dfSpotify_DataSet_Temp.sample(1000, random_state=42)

fltStatistic, fltPValue = stats.ttest_ind(
    dfSpotify_DataSet_Temp1, dfSpotify_DataSet_Temp2, equal_var=False, nan_policy="omit"
)
# print results
print(
    "Independent Parametric Test Results For Energy Column - Comparing Sample Of 1000 Records From Two Sample DataFrames"
)
print(
    "="
    * len(
        "Independent Parametric Test Results For Energy Column - Comparing Sample Of 1000 Records From Two Sample DataFrames"
    )
)
print(f"t-statistic: {fltStatistic:.4f}")
print(f"p-value: {fltPValue:.4f}")
print()

# run with the alternative set to greater to test if the mean of the first sample is greater than the second sample
# set equal_var to false as not assuming popuation variances are equal
fltTStatistic, fltPValue = stats.ttest_ind(
    dfSpotify_DataSet_Temp1,
    dfSpotify_DataSet_Temp2,
    equal_var=False,
    nan_policy="omit",
    alternative="greater",
)
# print results
print(
    "Independent Parametric Test Results For Energy Column - Test If Mean Of The First Sample Is Greater Than The Second Sample"
)
print(
    "="
    * len(
        "Independent Parametric Test Results For Energy Column  - Test If Mean Of The First Sample Is Greater Than The Second Sample"
    )
)
print(f"t-statistic: {fltTStatistic:.4f}")
print(f"p-value: {fltPValue:.4f}")
print()


# run T-Test for popularity column
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(
    by="popularity", ascending=False
).drop_duplicates(subset=["artists", "album_name", "track_name"], keep="first")[
    "popularity"
]

dfSpotify_DataSet_Temp1 = dfSpotify_DataSet_Temp.sample(1000, random_state=42)
dfSpotify_DataSet_Temp2 = dfSpotify_DataSet_Temp.sample(1000, random_state=42)

fltStatistic, fltPValue = stats.ttest_ind(
    dfSpotify_DataSet_Temp1, dfSpotify_DataSet_Temp2, equal_var=False, nan_policy="omit"
)
# print results
print(
    "Independent Parametric Test Results For Popularity Column - Comparing Sample Of 1000 Records From Two Sample DataFrames"
)
print(
    "="
    * len(
        "Independent Parametric Test Results For Popularity Column - Comparing Sample Of 1000 Records From Two Sample DataFrames"
    )
)
print(f"t-statistic: {fltStatistic:.4f}")
print(f"p-value: {fltPValue:.4f}")
print()

# run with the alternative set to greater to test if the mean of the first sample is greater than the second sample
# set equal_var to false as not assuming popuation variances are equal
fltTStatistic, fltPValue = stats.ttest_ind(
    dfSpotify_DataSet_Temp1,
    dfSpotify_DataSet_Temp2,
    equal_var=False,
    nan_policy="omit",
    alternative="greater",
)
# print results
print(
    "Independent Parametric Test Results For Popularity Column - Test If Mean Of The First Sample Is Greater Than The Second Sample"
)
print(
    "="
    * len(
        "Independent Parametric Test Results For Popularity Column - Test If Mean Of The First Sample Is Greater Than The Second Sample"
    )
)
print(f"t-statistic: {fltTStatistic:.4f}")
print(f"p-value: {fltPValue:.4f}")
print()

# now run independant T-Test for energy and popularity
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(
    by="popularity", ascending=False
).drop_duplicates(subset=["artists", "album_name", "track_name"], keep="first")

dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp[["popularity", "energy"]]

stat, p = ttest_ind(
    dfSpotify_DataSet_Temp["energy"],
    dfSpotify_DataSet_Temp["popularity"],
    nan_policy="omit",
)

print("Independent Parametric Test Results ComparingEnergy And Popularity Columns")
print(
    "="
    * len("Independent Parametric Test Results Comparing Energy And Popularity Columns")
)
print("t-statistic:", stat)
print("p-value:", p)
print()

# run Pearson correlation test for energy column
fltPearsonCorr, fltPearsonP = pearsonr(
    dfSpotify_DataSet_Temp["energy"], dfSpotify_DataSet_Temp["popularity"]
)
print("Pearson Correlation Test Results For Energy Against Popularity Column")
print(
    "*" * len("Pearson Correlation Test Results For Energy Against Popularity Column")
)
print(
    f"Pearson Correlation Coefficient: {fltPearsonCorr:.4f}, P-value: {fltPearsonP:.4f}"
)
print()


Independent Parametric Test Results For Energy Column - Comparing Sample Of 1000 Records From Two Sample DataFrames
t-statistic: 0.0000
p-value: 1.0000

Independent Parametric Test Results For Energy Column - Test If Mean Of The First Sample Is Greater Than The Second Sample
t-statistic: 0.0000
p-value: 0.5000

Independent Parametric Test Results For Popularity Column - Comparing Sample Of 1000 Records From Two Sample DataFrames
t-statistic: 0.0000
p-value: 1.0000

Independent Parametric Test Results For Popularity Column - Test If Mean Of The First Sample Is Greater Than The Second Sample
t-statistic: 0.0000
p-value: 0.5000

Independent Parametric Test Results ComparingEnergy And Popularity Columns
t-statistic: -473.84040619080037
p-value: 0.0

Pearson Correlation Test Results For Energy Against Popularity Column
*********************************************************************
Pearson Correlation Coefficient: 0.0129, P-value: 0.0001



# 

# Observations

Energy has a p-value of 1 in the first test which means it is not significant, in the second test the p-value was 0.5 which is also not significant as it is above 0.05.

Popularity has a p-value of 1 in the first test and like energy has a p-value of 0.5 in the second, so neither result is statistically significant.

The independent T-Test comparing energy *and* popularity produced a p-value very close to 0 which is statistically significant. However, this test compares the two variables rather than measuring the relationship between them, so the correlation tests are more useful for our hypothesis.

The Pearson correlation test yielded a p-value of 0.0001 which is statistically significant as it is below 0.05. However, the correlation value was only around 0.013, which means the actual relationship between energy and popularity is extremely weak.

# Conclusion

The Pearson correlation between energy and popularity was approximately 0.013 with a p-value of 0.0001.

As the p-value is below 0.05 we can reject hypothesis zero, so there is a statistically significant relationship between energy and popularity.

However, the correlation value is extremely close to zero which means the relationship itself is very weak and is not really significant in a practical sense.

The Spearman correlation was also very close to zero at around -0.017 which supports the same result.

Overall, although the relationship is statistically significant because of the large amount of data, energy does not appear to have much of a useful relationship with popularity.